# 02 — Preprocessing des données OpenAgenda

**Objectif** : Nettoyer et structurer les données brutes en vue du chunking.

**Entrée**  : `data/raw/openagenda_idf_YYYYMMDD.json`  
**Sortie**  : `data/processed/openagenda_idf_YYYYMMDD_clean.parquet`

**Étapes** :
1. Chargement & déduplication
2. Filtrage des événements sans description
3. Strip HTML via Docling
4. Construction de `date_display`
5. Sélection des champs finaux & sauvegarde

---

## 0. Imports & configuration

In [9]:
import json

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter
import pandas as pd

from notebooks.config import DATA_PROCESSED, DATA_RAW, REFERENCE_DATE

converter = DocumentConverter()


def html_to_text(html: str | None) -> str:
    if not isinstance(html, str) or html.strip() == "":
        return ""
    return converter.convert_string(html, format=InputFormat.HTML).document.export_to_markdown().strip()


DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

raw_files = sorted(DATA_RAW.glob("openagenda_îledefrance_*.json"))
raw_path = raw_files[-1]

print(f"Fichier source : {raw_path}")

Fichier source : /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/data/raw/openagenda_îledefrance_20260607.json


In [10]:
with open(raw_path, encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw["events"])
print(f"Chargé       : {len(df)} événements")


df = df.drop_duplicates(subset=["uid"], keep="first")
print(f"Après dedup  : {len(df)} événements")

Chargé       : 10000 événements
Après dedup  : 10000 événements


## 2. Filtrage des événements sans description

On supprime les événements sans `description_fr` **et** sans `longdescription_fr` — un chunk sans contenu textuel suffisant risque de provoquer des hallucinations du LLM.

Les événements sans `longdescription_fr` mais avec `description_fr` sont conservés — la description courte sera utilisée en fallback.

In [11]:
before = len(df)

mask_no_desc = (df["description_fr"].isna() | (df["description_fr"].str.strip() == "")) & (
    df["longdescription_fr"].isna() | (df["longdescription_fr"].str.strip() == "")
)

df = df[~mask_no_desc].copy()

print(
    f"Supprimés (aucune description) : {before - len(df)} ({(before - len(df)) / before * 100:.1f}%)"
)
print(f"Conservés : {len(df)}")

Supprimés (aucune description) : 160 (1.6%)
Conservés : 9840


## 3. Strip HTML via Docling

La `longdescription_fr` est du HTML brut (`<p>`, `<br>`, `<ul>`...). On utilise **Docling** pour convertir proprement en texte Markdown, bien plus fiable qu'une regex maison.

In [12]:
df["description_clean"] = df["longdescription_fr"].apply(html_to_text)

mask_empty_long = df["description_clean"] == ""
df.loc[mask_empty_long, "description_clean"] = df.loc[mask_empty_long, "description_fr"].fillna("")

print(f"Descriptions nettoyées : {(df['description_clean'] != '').sum()} / {len(df)}")
print(f"Fallback description_fr utilisé : {mask_empty_long.sum()}")

Clashing formatting: 'bold=True italic=True underline=False strikethrough=False script=<Script.BASELINE: 'baseline'>' and 'bold=True italic=False underline=False strikethrough=False script=<Script.BASELINE: 'baseline'>'! Chose 'bold=True italic=False underline=False strikethrough=False script=<Script.BASELINE: 'baseline'>'


Descriptions nettoyées : 9840 / 9840
Fallback description_fr utilisé : 489


In [13]:
sample = df[df["longdescription_fr"].notna()].iloc[0]

print("HTML brut :")
print(sample["longdescription_fr"][:])
print("\nTexte nettoyé :")
print(sample["description_clean"][:])

HTML brut :
<p>Cette visite-atelier invite petits et grands à découvrir l’exposition temporaire <em>La fabrique du Temps</em> de manière ludique et créative.<br>À travers une visite de l’exposition et un quiz pédagogique, les visiteurs explorent les outils et méthodes qui ont permis de capter le temps au fil des époques avant de fabriquer leur propre calendrier lunaire sous forme de pop-up ou de leporello, en fonction du niveau des participants.</p>

Texte nettoyé :
Cette visite-atelier invite petits et grands à découvrir l'exposition temporaire *La fabrique du Temps* de manière ludique et créative. À travers une visite de l'exposition et un quiz pédagogique, les visiteurs explorent les outils et méthodes qui ont permis de capter le temps au fil des époques avant de fabriquer leur propre calendrier lunaire sous forme de pop-up ou de leporello, en fonction du niveau des participants.


## 4. Construction de `date_display`

On construit une date lisible **avec année explicite** depuis les timestamps ISO,  
pour éviter l'ambiguïté de `daterange_fr` (ex: "Mardi 3 février, 15h00" sans année).

In [14]:
MONTHS_FR = {
    1: "janvier",
    2: "février",
    3: "mars",
    4: "avril",
    5: "mai",
    6: "juin",
    7: "juillet",
    8: "août",
    9: "septembre",
    10: "octobre",
    11: "novembre",
    12: "décembre",
}


def build_date_display(row) -> str:
    """Construit une date lisible avec année depuis les timestamps ISO."""
    try:
        first = pd.to_datetime(row["firstdate_begin"], utc=True)
        last = pd.to_datetime(row["lastdate_begin"], utc=True)

        def fmt(dt):
            return f"{dt.day} {MONTHS_FR[dt.month]} {dt.year} à {dt.strftime('%H:%M')}"

        if first.date() == last.date():
            return f"Le {fmt(first)}"
        else:
            return f"Du {fmt(first)} au {fmt(last)}"
    except Exception:
        return ""


df["date_display"] = df.apply(build_date_display, axis=1)

df[["daterange_fr", "date_display", "firstdate_begin", "lastdate_begin"]].sample(
    8, random_state=42
)

,daterange_fr,date_display,firstdate_begin,lastdate_begin
6912,16 - 19 octobre 2025,Du 16 octobre 2025 à 08:00 au 19 octobre 2025 ...,2025-10-16T08:00:00+00:00,2025-10-19T08:00:00+00:00
7237,"Mercredi 15 avril, 19h00",Le 15 avril 2026 à 17:00,2026-04-15T17:00:00+00:00,2026-04-15T17:00:00+00:00
106,20 et 21 septembre 2025,Du 20 septembre 2025 à 12:00 au 21 septembre 2...,2025-09-20T12:00:00+00:00,2025-09-21T12:00:00+00:00
452,23 et 24 mai,Du 23 mai 2026 à 13:00 au 24 mai 2026 à 14:15,2026-05-23T13:00:00+00:00,2026-05-24T14:15:00+00:00
8218,"Samedi 27 septembre 2025, 08h30",Le 27 septembre 2025 à 06:30,2025-09-27T06:30:00+00:00,2025-09-27T06:30:00+00:00
7680,19 et 20 mars,Du 19 mars 2026 à 18:00 au 20 mars 2026 à 18:00,2026-03-19T18:00:00+00:00,2026-03-20T18:00:00+00:00
8781,"Jeudi 28 mai, 19h00",Le 28 mai 2026 à 17:00,2026-05-28T17:00:00+00:00,2026-05-28T17:00:00+00:00
8519,"Samedi 20 septembre 2025, 09h00",Le 20 septembre 2025 à 07:00,2025-09-20T07:00:00+00:00,2025-09-20T07:00:00+00:00


## 5. Sélection des champs finaux & sauvegarde

In [15]:
FINAL_FIELDS = [
    # Identifiant
    "uid",
    # Contenu textuel
    "title_fr",
    "keywords_fr",
    "description_clean",
    "conditions_fr",
    # Date
    "date_display",
    "firstdate_begin",
    "lastdate_begin",
    # Localisation
    "location_name",
    "location_address",
    "location_city",
    "location_department",
    "location_region",
]

df_clean = df[FINAL_FIELDS].copy()

print(f"Shape final : {df_clean.shape}")
df_clean.head(3)

Shape final : (9840, 13)


,uid,title_fr,keywords_fr,description_clean,conditions_fr,date_display,firstdate_begin,lastdate_begin,location_name,location_address,location_city,location_department,location_region
0,6670838,Visite-atelier : Prendre la mesure du temps,None,Cette visite-atelier invite petits et grands à...,25 participants maximum.,Du 20 septembre 2025 à 12:30 au 21 septembre 2...,2025-09-20T12:30:00+00:00,2025-09-21T12:30:00+00:00,Musée de la poste,"34 Boulevard de Vaugirard, 75015 Paris, France",Paris,Paris,Île-de-France
1,96129611,La fresque de la Biodiversité : comprendre ce ...,None,Au travers d'un atelier ludique et collaborati...,Entrée gratuite à partir de 12 ans / Réservati...,Le 4 octobre 2025 à 07:30,2025-10-04T07:30:00+00:00,2025-10-04T07:30:00+00:00,Bibliothèque Jean-Jacques Sempe,30 Bis rue de la Libération 91680 Bruyères-le-...,Bruyères-le-Châtel,Essonne,Île-de-France
2,45384394,On met les voiles !,None,"A l'occasion de ""Rendez-vous aux jardins"", le ...",None,Du 7 juin 2025 à 08:00 au 8 juin 2025 à 08:00,2025-06-07T08:00:00+00:00,2025-06-08T08:00:00+00:00,Domaine national du Palais-Royal,"8 rue Montpensier, 75001 Paris",Paris,Paris,Île-de-France


In [16]:
timestamp = REFERENCE_DATE.replace("-", "")
output_path = DATA_PROCESSED / f"openagenda_idf_{timestamp}_clean.parquet"

df_clean.to_parquet(output_path, index=False)

print(f"Sauvegardé → {output_path}")
print(f"Taille    : {output_path.stat().st_size / 1024:.1f} Ko")
print(f"Lignes    : {len(df_clean)}")

Sauvegardé → /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/data/processed/openagenda_idf_20260607_clean.parquet
Taille    : 5415.9 Ko
Lignes    : 9840


---
➡️ **Suite** : `03.0-kl-chunking.ipynb` — construction des chunks avec en-tête structuré